Generate datasets for emotion concepts

In [ ]:
import json
import sys
import time
from dataclasses import asdict
from pathlib import Path
from typing import Iterable, Iterator, Sequence

import torch
import yaml

# Must come BEFORE the core.* imports below — this notebook lives two levels
# down, so `core` is only importable once src/ is on the path. Anchored to the
# project root, not the kernel's cwd, which Jupyter front-ends disagree about.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.corpus import (
    build_instruction,
    build_prompts,
    generate_corpus,
    generate_stories,
)
from core.shards import read_jsonl, read_shards, write_jsonl, write_metadata
from core.types import Prompt, Story
from core.utils import Model, mentions_emotion

/Users/folusoogunlana/code/oss/emotion-concepts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'core'

In [4]:
MODEL_PATH = "Qwen/Qwen2.5-0.5B-Instruct"

CACHE_DIR = PROJECT_ROOT / ".cache"
DATA_DIR = PROJECT_ROOT / "datasets" / "qwen-emotion-stories"
SEED = 0

model = Model(MODEL_PATH)
model.load_weights()
model.device

PosixPath('/Users/folusoogunlana/code/oss/emotion-concepts')

In [5]:
CONFIG_PATH = DATA_DIR / "config.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

EMOTIONS: tuple[str, ...] = tuple(CONFIG["emotions"])
TOPICS: tuple[str, ...] = tuple(CONFIG["topics"])
INTENSITIES: tuple[dict, ...] = tuple(CONFIG["intensity"])
IMPLICIT: tuple[dict, ...] = tuple(CONFIG["implicit"])

print(f"{len(EMOTIONS)} emotions, {len(TOPICS)} topics, "
      f"{len(INTENSITIES)} ladders, {len(IMPLICIT)} implicit scenarios")
print(f"corpus: {len(EMOTIONS) + 1} shards x {len(TOPICS)} topics x n_per_pair")

In [8]:
# inspect
prompts = build_prompts(EMOTIONS, TOPICS, n_per_pair=1)
print(f"{len(prompts)} prompts\n")
for p in prompts[:6]:
    print(f"[{p.emotion} / {p.topic} / {p.index}]\n{p.instruction}\n")

65 prompts

[joy / a train station / 1]
Write a short story (max 200 words) on a topic 'a train station' where a character experiences emotion 'joy'.

[joy / a job interview / 1]
Write a short story (max 200 words) on a topic 'a job interview' where a character experiences emotion 'joy'.

[joy / an old family recipe / 1]
Write a short story (max 200 words) on a topic 'an old family recipe' where a character experiences emotion 'joy'.

[joy / a broken bicycle / 1]
Write a short story (max 200 words) on a topic 'a broken bicycle' where a character experiences emotion 'joy'.

[joy / the last day of school / 1]
Write a short story (max 200 words) on a topic 'the last day of school' where a character experiences emotion 'joy'.

[sadness / a train station / 1]
Write a short story (max 200 words) on a topic 'a train station' where a character experiences emotion 'sadness'.



In [9]:
resp = model.gen(prompts[:2], sample=True)

In [10]:
resp[0]

"Once upon a time, in the bustling city of Tokyo, there was a young man named Masahiko who had just moved to the city after years of traveling. The city Hurricanes of Tokyo was his first home and his second place to call family.\nOne day, as Masahiko was walking through Central Station, he noticed a group of people gathered around a little girl with a bright smile. The girl's face lit up with joy, and she waved excitedly at Masahiko.\nAs he watched her smiling, he felt a sense of excitement about this unexpected moment. The experience of feeling joyful and happy was like nothing else in the world.\n\nFrom that moment on, whenever he saw people celebrating joyfully, or when someone had a heart full of hope or optimism, it made him feel good inside.\nThe trains started moving again one morning. As he stood on the platform, watching the crowd gather, he couldn't help but feel errorful. He was filled with an overwhelming sense of contentment and happiness. It wasn’t until later that he und

In [12]:
for s in generate_stories(model, prompts[:5]):
    print(s.text)

5/5 44.2s
In the heart of a bustling city, there stood a quaint little train station known as Harmony Lane. The station was a harmonious blend of old and new - the old trams that clanked along the tracks, the neon lights that flickered at the platforms, and the chirping birds overhead.

One crisp autumn morning, Emma, an elderly woman with long strawberry-blonde hair and piercing blue eyes, walked through the doors of Harmony Lane. She had just received her weekly allowance from the local bank الحكمية which made her job easier in the bustling city of Zephyria.

As she entered the station, the air was already filled with the smell of spices from around the world. The aroma of freshly baked pastries wafted across the platform, enticing Emma to try them. Her mind raced with thoughts of how much better she could have been without needing money to survive the day.

Emma's curiosity brought her to the ticket counter. It was here that she would buy her weekly allowance. As she reached out for

## Shards

IO lives in `core/shards.py`. One file per emotion, written atomically, so a crashed run costs one emotion and the resume check is just "does this file exist".

## Run

`limit` writes `<emotion>.sample` shards, which `read_shards` filters out — so the eyeball pass cannot contaminate the corpus.

In [ ]:
# eyeball pass — 2 stories per emotion, written to *.sample shards
# (read_shards filters those out, so it cannot contaminate the corpus)
generate_corpus(model, DATA_DIR, emotions=EMOTIONS, topics=TOPICS, limit=2)

2/2 27.0s
wrote joy: 2
2/2 18.2s
wrote sadness: 2
2/2 18.5s
wrote anger: 2
2/2 18.9s
wrote fear: 2
2/2 18.3s
wrote disgust: 2
2/2 22.4s
wrote surprise: 2
2/2 39.3s
wrote neutral: 2


{'joy': 2,
 'sadness': 2,
 'anger': 2,
 'fear': 2,
 'disgust': 2,
 'surprise': 2,
 'calm': 2,
 'desperation': 2,
 'pride': 2,
 'shame': 2,
 'loneliness': 2,
 'excitement': 2,
 'neutral': 2}

In [ ]:
# the real run — 1,300 stories. Prefer the terminal for this:
#   caffeinate -i uv run python src/scripts/generate_corpus.py
counts = generate_corpus(model, DATA_DIR, emotions=EMOTIONS, topics=TOPICS, n_per_pair=4)
write_metadata(DATA_DIR, model=MODEL_PATH, seed=SEED, batch_size=16,
               n_per_pair=4, emotions=list(EMOTIONS), topics=list(TOPICS), counts=counts)
counts

## Publish

The notebook's job ends with the shards. `build_dataset.py` assigns the split,
derives the held-out sets from `config.yaml`, and renders `README.md`; `hf upload`
syncs the folder. The folder is the Hub repo — what you see locally is published.

```bash
uv run python src/scripts/build_dataset.py
hf upload foogunlana/qwen-emotion-stories datasets/qwen-emotion-stories \
    --repo-type=dataset --delete "*"
```